<p align="center">
  <img src="https://seekvectorlogo.com/wp-content/uploads/2018/01/kayak-vector-logo.png" width="200">
</p>

# Plan your trip with Kayak
**Auteur :** Aïcha FATHELLAH : Formation Fullstack Data Engineering, Jedha  
**Objectif :** Recommander les 5 meilleures destinations françaises et les 20 meilleurs hôtels associés, en combinant données météo (API OpenWeatherMap) et données hôtelières (scraping Booking.com), stockées sur AWS S3 / RDS.

---

**Pipeline :**  
`Géocodage` → `API Météo` → `Scraping Hôtels` → `Nettoyage & Scoring` → `S3 (Data Lake)` → `RDS (Data Warehouse)` → `Visualisation`

## 1. Installation & Imports

In [1]:
%pip install selenium webdriver-manager boto3 sqlalchemy psycopg2-binary plotly geopy python-dotenv -q

import json, time, re, os
from io import StringIO
from pathlib import Path

import pandas as pd
import numpy as np
import requests
import plotly.express as px

from geopy.geocoders import Nominatim
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
import boto3
from sqlalchemy import create_engine
from dotenv import load_dotenv

print("Environnement prêt")


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Environnement prêt


## 2. Géocodage des 35 destinations (Nominatim)
Récupération des coordonnées GPS via l'API OpenStreetMap : pause d'1 s entre chaque ville pour respecter le rate limit.

In [2]:
villes = [
    "Mont Saint Michel", "St Malo", "Bayeux", "Le Havre", "Rouen", "Paris",
    "Amiens", "Lille", "Strasbourg", "Chateau de Chambord", "Orleans", "Avallon",
    "Dijon", "Besancon", "Annecy", "Grenoble", "Lyon", "Gorges du Verdon",
    "Bormes les Mimosas", "Cassis", "Marseille", "Aix en Provence", "Avignon",
    "Uzès", "Nimes", "Aigues Mortes", "Saintes Maries de la Mer", "Collioure",
    "Carcassonne", "Ariege", "Toulouse", "Montauban", "Biarritz", "Bayonne", "La Rochelle"
]

geolocator = Nominatim(user_agent="kayak_project_agent")
villes_data = []

for ville in villes:
    try:
        loc = geolocator.geocode(f"{ville}, France")
        if loc:
            villes_data.append({"city": ville, "latitude": loc.latitude, "longitude": loc.longitude})
            print(f"{ville}")
        else:
            print(f"{ville} — non trouvé")
        time.sleep(1)
    except Exception as e:
        print(f"Erreur {ville}: {e}")

df_villes = pd.DataFrame(villes_data)
df_villes.insert(0, "city_id", range(1, len(df_villes) + 1))
df_villes.to_csv("villes_coords.csv", index=False)
display(df_villes.head())

Mont Saint Michel
St Malo
Bayeux
Le Havre
Rouen
Paris
Amiens
Lille
Strasbourg
Chateau de Chambord
Orleans
Avallon
Dijon
Besancon
Annecy
Grenoble
Lyon
Gorges du Verdon
Bormes les Mimosas
Cassis
Marseille
Aix en Provence
Avignon
Uzès
Nimes
Aigues Mortes
Saintes Maries de la Mer
Collioure
Carcassonne
Ariege
Toulouse
Montauban
Biarritz
Bayonne
La Rochelle


,city_id,city,latitude,longitude
0,1,Mont Saint Michel,48.635954,-1.511460
1,2,St Malo,48.649518,-2.026041
2,3,Bayeux,49.276462,-0.702474
3,4,Le Havre,49.493898,0.107973
4,5,Rouen,49.440459,1.093966


## 3. Données météo : API OpenWeatherMap
Clé API chargée depuis `config/openweather.env` (variable `OPENWEATHER_API_KEY`).  
Endpoint : `data/2.5/forecast` : prévisions 5 j / 3 h, on retient le premier créneau par ville.

In [3]:
load_dotenv(dotenv_path=Path("config/openweather.env"))
API_KEY = os.getenv("OPENWEATHER_API_KEY")

weather_results = []

for _, row in df_villes.iterrows():
    url = (
        f"https://api.openweathermap.org/data/2.5/forecast"
        f"?lat={row['latitude']}&lon={row['longitude']}"
        f"&appid={API_KEY}&units=metric&lang=fr"
    )
    resp = requests.get(url)
    if resp.status_code == 200:
        first = resp.json()["list"][0]
        weather_results.append({
            "city_id": row["city_id"],
            "city":    row["city"],
            "temp":    first["main"]["temp"],
            "rain_prob": first.get("pop", 0),
            "weather_description": first["weather"][0]["description"]
        })
        print(f"{row['city']}")
    else:
        print(f"{row['city']} — code {resp.status_code}")
    time.sleep(0.2)

df_weather_final = pd.merge(
    df_villes,
    pd.DataFrame(weather_results)[["city_id", "temp", "rain_prob", "weather_description"]],
    on="city_id"
)
df_weather_final.to_csv("villes_meteo_coords.csv", index=False)
display(df_weather_final.head())

Mont Saint Michel
St Malo
Bayeux
Le Havre
Rouen
Paris
Amiens
Lille
Strasbourg
Chateau de Chambord
Orleans
Avallon
Dijon
Besancon
Annecy
Grenoble
Lyon
Gorges du Verdon
Bormes les Mimosas
Cassis
Marseille
Aix en Provence
Avignon
Uzès
Nimes
Aigues Mortes
Saintes Maries de la Mer
Collioure
Carcassonne
Ariege
Toulouse
Montauban
Biarritz
Bayonne
La Rochelle


,city_id,city,latitude,longitude,temp,rain_prob,weather_description
0,1,Mont Saint Michel,48.635954,-1.511460,10.58,1.00,pluie modérée
1,2,St Malo,48.649518,-2.026041,11.82,1.00,légère pluie
2,3,Bayeux,49.276462,-0.702474,12.17,0.37,légère pluie
3,4,Le Havre,49.493898,0.107973,11.67,0.00,nuageux
4,5,Rouen,49.440459,1.093966,13.73,0.00,nuageux


## 4. Scraping Booking.com : Selenium (Chrome headless)
`requests` bloqué par Booking (code 202 + JS dynamique) → utilisation de Selenium.  
Stratégie : scroll progressif pour déclencher le lazy-loading, limite à 20 hôtels par ville, checkpoint CSV après chaque ville.

In [4]:
def scrape_hotels_selenium(city):
    """Retourne une liste de dicts {city, hotel_name, url, rating, description}."""
    opts = Options()
    opts.add_argument("--headless")
    opts.add_argument("--window-size=1920,1080")
    opts.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                      "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36")

    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=opts)
    hotels = []
    url = f"https://www.booking.com/searchresults.fr.html?ss={city.replace(' ', '%20')}"

    try:
        driver.get(url)
        time.sleep(5)
        for i in range(4):
            driver.execute_script(f"window.scrollTo(0, {(i+1)*1000});")
            time.sleep(2)

        for card in driver.find_elements(By.CSS_SELECTOR, '[data-testid="property-card"]')[:20]:
            try:
                name     = card.find_element(By.CSS_SELECTOR, '[data-testid="title"]').text.strip()
                hotel_url = card.find_element(By.CSS_SELECTOR, '[data-testid="title-link"]').get_attribute("href")
                try:
                    raw   = card.find_element(By.CSS_SELECTOR, '[data-testid="review-score"]').text
                    m     = re.search(r"(\d[,\.]\d)", raw)
                    rating = float(m.group(1).replace(",", ".")) if m else 0.0
                except:
                    rating = 0.0
                try:
                    desc = card.find_element(By.CSS_SELECTOR, '[data-testid="recommended-units"]').text.strip()
                except:
                    desc = "N/A"
                if name:
                    hotels.append({"city": city, "hotel_name": name, "url": hotel_url,
                                   "rating": rating, "description": desc})
            except:
                continue
    finally:
        driver.quit()
    return hotels

In [5]:
OUTPUT_FILE = "hotels_booking_35_villes.csv"
final_data  = []

for _, row in df_villes.iterrows():
    print(f"[{row['city_id']}/35] {row['city']}...")
    try:
        results = scrape_hotels_selenium(row["city"])
        for h in results:
            h["city_id"] = row["city_id"]
        final_data.extend(results)
        pd.DataFrame(final_data).to_csv(OUTPUT_FILE, index=False)   # checkpoint
        print(f"  → {len(results)} hôtels | total cumulé : {len(final_data)}")
    except Exception as e:
        print(f"Erreur : {e}")
    time.sleep(5)

df_hotels_raw = pd.DataFrame(final_data)
print(f"\nScraping terminé — {len(df_hotels_raw)} lignes")
display(df_hotels_raw.head())

[1/35] Mont Saint Michel...
  → 3 hôtels | total cumulé : 3
[2/35] St Malo...
  → 3 hôtels | total cumulé : 6
[3/35] Bayeux...
  → 3 hôtels | total cumulé : 9
[4/35] Le Havre...
  → 3 hôtels | total cumulé : 12
[5/35] Rouen...
  → 3 hôtels | total cumulé : 15
[6/35] Paris...
  → 3 hôtels | total cumulé : 18
[7/35] Amiens...
  → 3 hôtels | total cumulé : 21
[8/35] Lille...
  → 3 hôtels | total cumulé : 24
[9/35] Strasbourg...
  → 3 hôtels | total cumulé : 27
[10/35] Chateau de Chambord...
  → 3 hôtels | total cumulé : 30
[11/35] Orleans...
  → 3 hôtels | total cumulé : 33
[12/35] Avallon...
  → 3 hôtels | total cumulé : 36
[13/35] Dijon...
  → 3 hôtels | total cumulé : 39
[14/35] Besancon...
  → 4 hôtels | total cumulé : 43
[15/35] Annecy...
  → 3 hôtels | total cumulé : 46
[16/35] Grenoble...
  → 3 hôtels | total cumulé : 49
[17/35] Lyon...
  → 3 hôtels | total cumulé : 52
[18/35] Gorges du Verdon...
  → 3 hôtels | total cumulé : 55
[19/35] Bormes les Mimosas...
  → 3 hôtels | total cu

,city,hotel_name,url,rating,description,city_id
0,Mont Saint Michel,Le Mouton Blanc,https://www.booking.com/hotel/fr/le-mouton-bla...,7.3,N/A,1
1,Mont Saint Michel,Le Relais Du Roy,https://www.booking.com/hotel/fr/le-relais-du-...,8.3,N/A,1
2,Mont Saint Michel,Hôtel la Croix Blanche,https://www.booking.com/hotel/fr/ha-el-la-croi...,8.2,N/A,1
3,St Malo,Le Nid Cosy - Studio proche de la plage,https://www.booking.com/hotel/fr/le-nid-studio...,8.5,N/A,2
4,St Malo,L'Emeraude,https://www.booking.com/hotel/fr/l-39-emeraude...,9.4,N/A,2


## 5. Nettoyage, agrégation et calcul du score de voyage
**Nettoyage :** suppression des lignes sans note, déduplication par `(hotel_name, city)`.  
**Score de voyage :** `0.5 × temp + 0.5 × (mean_hotel_rating × 2)` — pondération égale météo / qualité hôtelière.

In [6]:
# ── Nettoyage ──────────────────────────────────────────────────────────────────
df_clean = pd.read_csv("hotels_booking_35_villes.csv").copy()
df_clean = (df_clean
    .dropna(subset=["rating"])
    .drop_duplicates(subset=["hotel_name", "city"])
)
df_clean["description"] = df_clean["description"].fillna("N/A")
df_clean.to_csv("hotels_cleaned_full.csv", index=False)

# ── Score agrégé par ville ─────────────────────────────────────────────────────
df_city_scores = (df_clean
    .groupby(["city_id", "city"])["rating"]
    .mean().round(2)
    .reset_index()
    .rename(columns={"rating": "mean_hotel_rating"})
)
df_city_scores.to_csv("villes_scores_aggregated.csv", index=False)

# ── Fusion météo + hôtels ──────────────────────────────────────────────────────
df_weather = pd.read_csv("villes_meteo_coords.csv")
df_final = pd.merge(df_weather, df_city_scores, on=["city_id", "city"], how="left")
df_final["travel_score"] = (df_final["temp"] * 0.5) + (df_final["mean_hotel_rating"] * 2 * 0.5)
df_final = df_final.sort_values("travel_score", ascending=False)
df_final.to_csv("kayak_final_enriched_data.csv", index=False)

print(f"Dataset final : {df_final.shape}")
display(df_final.head(10))

Dataset final : (35, 9)


,city_id,city,latitude,longitude,temp,rain_prob,weather_description,mean_hotel_rating,travel_score
19,20,Cassis,43.214036,5.539632,19.37,0.00,ciel dégagé,9.47,19.155
18,19,Bormes les Mimosas,43.150697,6.341928,18.22,0.00,ciel dégagé,9.50,18.610
21,22,Aix en Provence,43.529842,5.447474,18.47,0.00,ciel dégagé,8.53,17.765
22,23,Avignon,43.949249,4.805901,18.81,0.77,ciel dégagé,8.13,17.535
25,26,Aigues Mortes,43.566152,4.191540,16.88,0.00,ciel dégagé,9.00,17.440
24,25,Nimes,43.837425,4.360069,17.74,0.43,ciel dégagé,8.20,17.070
16,17,Lyon,45.757814,4.832011,17.03,1.00,pluie modérée,8.43,16.945
31,32,Montauban,44.017584,1.354999,15.35,1.00,légère pluie,9.10,16.775
14,15,Annecy,45.899235,6.128885,16.51,1.00,légère pluie,8.50,16.755
8,9,Strasbourg,48.584614,7.750713,16.56,1.00,légère pluie,8.47,16.750


## 6. Data Lake : AWS S3
Clés AWS chargées depuis `config/aws.env`.  
Les 4 fichiers CSV sont envoyés dans le bucket S3 (Data Lake — couche brute + transformée).

In [ ]:
load_dotenv(dotenv_path=Path("config/aws.env"), override=True)

ACCESS_KEY  = os.getenv("AWS_ACCESS_KEY_ID")
SECRET_KEY  = os.getenv("AWS_SECRET_ACCESS_KEY")
BUCKET_NAME = os.getenv("S3_BUCKET_NAME")
REGION      = os.getenv("AWS_REGION", "eu-west-3")

session  = boto3.Session(aws_access_key_id=ACCESS_KEY,
                          aws_secret_access_key=SECRET_KEY,
                          region_name=REGION)
s3       = session.resource("s3")
s3_client = session.client("s3")

# Création du bucket si inexistant
try:
    s3_client.head_bucket(Bucket=BUCKET_NAME)
    print(f"Bucket '{BUCKET_NAME}' déjà existant.")
except:
    s3.create_bucket(Bucket=BUCKET_NAME,
                     CreateBucketConfiguration={"LocationConstraint": REGION})
    print(f"Bucket '{BUCKET_NAME}' créé.")

# Upload
for fname in ["villes_meteo_coords.csv", "hotels_cleaned_full.csv",
              "villes_scores_aggregated.csv", "kayak_final_enriched_data.csv"]:
    if os.path.exists(fname):
        s3.Bucket(BUCKET_NAME).upload_file(fname, fname)
        print(f"{fname} envoyé")
    else:
        print(f"{fname} non trouvé localement")

## 7. Data Warehouse : AWS RDS (PostgreSQL)
Clés chargées depuis `config/aws.env`.  
Deux tables relationnelles : `cities` (1 ligne / ville) et `hotels` (N lignes / ville, liées par `city_id`).

In [ ]:
load_dotenv(dotenv_path=Path("config/aws.env"), override=True)

DB_USER     = os.getenv("RDS_USERNAME")
DB_PASSWORD = os.getenv("RDS_PASSWORD")
DB_HOST     = os.getenv("RDS_HOSTNAME")
DB_PORT     = os.getenv("RDS_PORT", "5432")
DB_NAME     = os.getenv("RDS_DB_NAME", "postgres")

engine = create_engine(f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

try:
    df_final.to_sql("cities", con=engine, if_exists="replace", index=False)
    print("Table 'cities' injectée (35 destinations)")

    df_clean.to_sql("hotels", con=engine, if_exists="replace", index=False)
    print(f"Table 'hotels' injectée ({len(df_clean)} établissements)")
except Exception as e:
    print(f"Erreur RDS : {e}")
    print("→ Vérifier le Security Group AWS (port 5432, IP autorisée)")

## 8. Visualisation : Cartes interactives Plotly
### Carte 1 : Top 5 destinations (score météo + hôtels)

In [11]:
# Tableau Top 5 destinations
top_5_display = df_final.head(5)[["city", "temp", "rain_prob", "mean_hotel_rating", "travel_score"]].copy()
top_5_display.columns = ["Destination", "Température (°C)", "Prob. pluie", "Note hôtels", "Score voyage"]
top_5_display = top_5_display.round(2).reset_index(drop=True)
top_5_display.index += 1
display(top_5_display)

# Carte interactive des 5 meilleures destinations
top_5 = df_final.head(5)

fig = px.scatter_mapbox(
    top_5,
    lat="latitude", lon="longitude",
    hover_name="city",
    hover_data={"latitude": False, "longitude": False,
                "temp": ":.1f", "rain_prob": ":.1f", "mean_hotel_rating": ":.1f"},
    color="temp",
    size="travel_score",
    color_continuous_scale=px.colors.sequential.YlOrRd,
    size_max=18, zoom=5,
    mapbox_style="open-street-map",
    title="Top 5 Destinations — Meilleur compromis Météo / Hôtels"
)
fig.update_layout(margin={"r":10,"t":50,"l":10,"b":10})
fig.show()

,Destination,Température (°C),Prob. pluie,Note hôtels,Score voyage
1,Cassis,19.37,0.00,9.47,19.16
2,Bormes les Mimosas,18.22,0.00,9.50,18.61
3,Aix en Provence,18.47,0.00,8.53,17.76
4,Avignon,18.81,0.77,8.13,17.54
5,Aigues Mortes,16.88,0.00,9.00,17.44


### Carte 2 : Top 20 hôtels les mieux notés

In [10]:
# Tableau Top 20 hôtels
df_coords = df_weather[["city", "latitude", "longitude"]]
top_20 = (pd.merge(df_clean, df_coords, on="city", how="left")
            .sort_values("rating", ascending=False)
            .head(20))

top_20_display = top_20[["hotel_name", "city", "rating", "url"]].copy()
top_20_display.columns = ["Hôtel", "Ville", "Note", "URL"]
top_20_display = top_20_display.reset_index(drop=True)
top_20_display.index += 1
display(top_20_display)

# Carte interactive des 20 meilleurs hôtels
df_coords = df_weather[["city", "latitude", "longitude"]]
top_20 = (pd.merge(df_clean, df_coords, on="city", how="left")
            .sort_values("rating", ascending=False)
            .head(20))

fig2 = px.scatter_mapbox(
    top_20,
    lat="latitude", lon="longitude",
    hover_name="hotel_name",
    hover_data={"latitude": False, "longitude": False,
                "city": True, "rating": ":.1f"},
    color="rating", size="rating",
    color_continuous_scale=px.colors.sequential.Viridis,
    zoom=5, mapbox_style="open-street-map",
    title="<b>Top 20 Hôtels les mieux notés en France</b>"
)
fig2.update_layout(margin={"r":10,"t":50,"l":10,"b":10})
fig2.show()

,Hôtel,Ville,Note,URL
1,LE SEPT charmant studio aux portes des calanques,Cassis,9.9,https://www.booking.com/hotel/fr/le-sept-charm...
2,"L'Odyssée provençale, un duplex de charme de 2...",Bormes les Mimosas,9.8,https://www.booking.com/hotel/fr/duplex-de-cha...
3,Studio Le sillon malouin vue mer,St Malo,9.8,https://www.booking.com/hotel/fr/studio-vue-me...
4,La DAME de FLAUX,Uzès,9.8,https://www.booking.com/hotel/fr/la-dame-de-fl...
5,Chambre d'hôte Montlivault / Chambord,Chateau de Chambord,9.7,https://www.booking.com/hotel/fr/chambre-d-hot...
6,"room five - parking, balnéothérapie, place nat...",Montauban,9.6,https://www.booking.com/hotel/fr/room-five-mon...
7,Appartement Aloa,Carcassonne,9.5,https://www.booking.com/hotel/fr/aloa.fr.html?...
8,Château de Monceaux 5mn de Bayeux proche Mer,Bayeux,9.5,https://www.booking.com/hotel/fr/le-chateau-de...
9,L'Emeraude,St Malo,9.4,https://www.booking.com/hotel/fr/l-39-emeraude...
10,Annecy Centre Historique - 165 m - 3 chambres ...,Annecy,9.4,https://www.booking.com/hotel/fr/appartement-d...


## 9. Conclusion :

Ce projet met en œuvre un pipeline ETL complet :

| Étape | Technologie |
|---|---|
| Géocodage | Nominatim (OpenStreetMap) |
| Données météo | API OpenWeatherMap |
| Scraping hôtels | Selenium + BeautifulSoup |
| Stockage brut | AWS S3 (Data Lake) |
| Stockage structuré | AWS RDS / PostgreSQL (Data Warehouse) |
| Visualisation | Plotly Express (Mapbox) |

**Piste d'amélioration :** automatisation hebdomadaire via AWS Lambda + enrichissement avec les prix des billets de train (API SNCF) pour un score de voyage plus complet.